# python-dlshogi2 Training Notebook

This notebook trains a shogi policy-value network using MCTS (AlphaGo Zero approach).

**Runtime:** Go to `Runtime > Change runtime type` and select **GPU** (T4 recommended).

## 1. Install dependencies

In [1]:
!pip install cshogi
!pip install onnxscript
!pip install git+https://github.com/LoveKapibarasan/python-dlshogi2.git

In [ ]:
# Restart the runtime so newly installed packages are importable
import os
os.kill(os.getpid(), 9)

## 2. Check GPU

In [2]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 3. Mount Google Drive (recommended for large datasets)

Mount Drive to read training data and save checkpoints persistently.
Skip this cell if you are uploading data directly.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Update these paths to match your Drive layout
TRAIN_DATA  = '/content/drive/MyDrive/shogi/train.hcpe'
TEST_DATA   = '/content/drive/MyDrive/shogi/test.hcpe'
CHECKPOINT  = '/content/drive/MyDrive/shogi/checkpoints/checkpoint-{epoch:03}.pth'
RESUME_FROM = ''  # e.g. '/content/drive/MyDrive/shogi/checkpoints/checkpoint-001.pth'

Mounted at /content/drive


### (Alternative) Upload data files directly

If you prefer not to use Drive, upload `.hcpe` files here instead.
Note: uploaded files are lost when the session ends.

In [ ]:
# Uncomment to upload files manually
# from google.colab import files
# uploaded = files.upload()  # select train.hcpe and test.hcpe

# TRAIN_DATA  = 'train.hcpe'
# TEST_DATA   = 'test.hcpe'
# CHECKPOINT  = 'checkpoints/checkpoint-{epoch:03}.pth'
# RESUME_FROM = ''

## 4. Convert CSA game records to HCPE (optional)

Skip this section if you already have `.hcpe` files.

Download CSA game records (e.g. from [Floodgate](http://wdoor.c.u-tokyo.ac.jp/shogi/)), unzip them, then run the converter.

In [ ]:
# Example: convert a directory of CSA files
# !python -m utils.csa_to_hcpe \
#     /content/csa_dir \
#     /content/train.hcpe \
#     /content/test.hcpe \
#     --filter_moves 50 \
#     --filter_rating 3500 \
#     --test_ratio 0.1

## 5. Training configuration

In [ ]:
# --- Edit these settings ---
GPU_ID        = 0       # -1 for CPU
EPOCHS        = 5
BATCH_SIZE    = 1024
LEARNING_RATE = 0.01
EVAL_INTERVAL = 100     # log every N steps
# ---------------------------

import os
os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)

print('Train data :', TRAIN_DATA)
print('Test data  :', TEST_DATA)
print('Checkpoint :', CHECKPOINT)
print('Resume from:', RESUME_FROM or '(none)')

## 6. Run training

In [ ]:
import torch
import torch.optim as optim
import logging

from pydlshogi2.network.policy_value_resnet import PolicyValueNetwork
from pydlshogi2.dataloader import HcpeDataLoader

logging.basicConfig(
    format='%(asctime)s\t%(levelname)s\t%(message)s',
    datefmt='%Y/%m/%d %H:%M:%S',
    level=logging.INFO,
)

device = torch.device(f'cuda:{GPU_ID}') if GPU_ID >= 0 else torch.device('cpu')

model = PolicyValueNetwork()
model.to(device)

optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=0.0001)
cross_entropy_loss   = torch.nn.CrossEntropyLoss()
bce_with_logits_loss = torch.nn.BCEWithLogitsLoss()

epoch = 0
t     = 0

if RESUME_FROM:
    logging.info(f'Resuming from {RESUME_FROM}')
    ckpt = torch.load(RESUME_FROM, map_location=device)
    epoch = ckpt['epoch']
    t     = ckpt['t']
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    optimizer.param_groups[0]['lr'] = LEARNING_RATE

logging.info('Loading training data')
train_loader = HcpeDataLoader(TRAIN_DATA, BATCH_SIZE, device, shuffle=True)
logging.info('Loading test data')
test_loader  = HcpeDataLoader(TEST_DATA,  BATCH_SIZE, device)
logging.info(f'train positions: {len(train_loader):,}')
logging.info(f'test  positions: {len(test_loader):,}')

def accuracy(y, t):
    return (torch.max(y, 1)[1] == t).sum().item() / len(t)

def binary_accuracy(y, t):
    return (y >= 0).eq(t >= 0.5).sum().item() / len(t)

def save_checkpoint(epoch, t):
    path = CHECKPOINT.format(epoch=epoch, step=t)
    torch.save({'epoch': epoch, 't': t,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict()}, path)
    logging.info(f'Checkpoint saved: {path}')


for _ in range(EPOCHS):
    epoch += 1
    steps_interval = sum_lp_interval = sum_lv_interval = 0
    steps_epoch    = sum_lp_epoch    = sum_lv_epoch    = 0

    for x, move_label, result in train_loader:
        model.train()
        y1, y2 = model(x)
        loss_p  = cross_entropy_loss(y1, move_label)
        loss_v  = bce_with_logits_loss(y2, result)
        optimizer.zero_grad()
        (loss_p + loss_v).backward()
        optimizer.step()
        t += 1

        steps_interval  += 1
        sum_lp_interval += loss_p.item()
        sum_lv_interval += loss_v.item()

        if t % EVAL_INTERVAL == 0:
            model.eval()
            with torch.no_grad():
                tx, tm, tr = test_loader.sample()
                ty1, ty2   = model(tx)
                tlp = cross_entropy_loss(ty1, tm).item()
                tlv = bce_with_logits_loss(ty2, tr).item()
                tap = accuracy(ty1, tm)
                tav = binary_accuracy(ty2, tr)

            logging.info(
                f'epoch={epoch} step={t} '
                f'train_loss={sum_lp_interval/steps_interval:.5f}+{sum_lv_interval/steps_interval:.5f} '
                f'test_loss={tlp:.5f}+{tlv:.5f} '
                f'test_acc policy={tap:.4f} value={tav:.4f}'
            )

            steps_epoch  += steps_interval;  sum_lp_epoch += sum_lp_interval;  sum_lv_epoch += sum_lv_interval
            steps_interval = sum_lp_interval = sum_lv_interval = 0

    steps_epoch  += steps_interval;  sum_lp_epoch += sum_lp_interval;  sum_lv_epoch += sum_lv_interval

    # Full test-set evaluation at end of epoch
    test_steps = tlp_sum = tlv_sum = tap_sum = tav_sum = 0
    model.eval()
    with torch.no_grad():
        for x, move_label, result in test_loader:
            y1, y2   = model(x)
            test_steps += 1
            tlp_sum    += cross_entropy_loss(y1, move_label).item()
            tlv_sum    += bce_with_logits_loss(y2, result).item()
            tap_sum    += accuracy(y1, move_label)
            tav_sum    += binary_accuracy(y2, result)

    logging.info(
        f'[epoch {epoch} summary] '
        f'train_loss_avr={sum_lp_epoch/steps_epoch:.5f}+{sum_lv_epoch/steps_epoch:.5f} '
        f'test_loss={tlp_sum/test_steps:.5f}+{tlv_sum/test_steps:.5f} '
        f'test_acc policy={tap_sum/test_steps:.4f} value={tav_sum/test_steps:.4f}'
    )

    save_checkpoint(epoch, t)

## 7. Export to ONNX (optional)

Export the trained model to ONNX format for use with `onnx_player`.

In [ ]:
import torch
from pydlshogi2.network.policy_value_resnet import PolicyValueNetwork
from pydlshogi2.features import FEATURES_NUM

CHECKPOINT_TO_EXPORT = CHECKPOINT.format(epoch=epoch, step=t)
ONNX_OUTPUT          = CHECKPOINT_TO_EXPORT.replace('.pth', '.onnx')

ckpt  = torch.load(CHECKPOINT_TO_EXPORT, map_location='cpu')
model = PolicyValueNetwork()
model.load_state_dict(ckpt['model'])
model.eval()

dummy = torch.zeros(1, FEATURES_NUM, 9, 9)
torch.onnx.export(
    model, dummy, ONNX_OUTPUT,
    input_names=['input'],
    output_names=['policy', 'value'],
    dynamic_axes={'input': {0: 'batch'}, 'policy': {0: 'batch'}, 'value': {0: 'batch'}},
    opset_version=11,
)
print('Exported:', ONNX_OUTPUT)

## 8. Download checkpoint

Download the trained model to your local machine (only needed if you did not use Drive).

In [ ]:
from google.colab import files
files.download(CHECKPOINT.format(epoch=epoch, step=t))